# 01 — Functions on Shapes


## Intuition

Think of a function on a shape as a **scalar field**: temperature at every point of a surface, terrain height, the distance from the tip of a nose. Mathematically, it is simply a map

$$f : M \to \mathbb{R}$$

that assigns one real number to every point of the shape. On a triangle mesh or a point cloud, we store this as a vector $\mathbf{f} \in \mathbb{R}^n$ — one value per vertex.

In spectral geometry, we study the geometry of the shapes moving our focus from points, edges,faces, to functions defined on the domain, and how the operators act on these.

In [1]:
import gsops.backend as gs
import numpy as np
import polyscope as ps

import geomfum.linalg as la
from geomfum.dataset import NotebooksDataset
from geomfum.shape import TriangleMesh

ps.init()

In [2]:
# load a shape
dataset = NotebooksDataset()
mesh = TriangleMesh.from_file(dataset.get_filename("cat-00"))

V = np.asarray(mesh.vertices)  # (n, 3)
F = np.asarray(mesh.faces)  # (m, 3)
n = mesh.n_vertices
print(f"Cat: {n} vertices, {mesh.n_faces} faces")

Cat: 7207 vertices, 14410 faces


## Step 1 — define some functions and look at them

Before any math, let us build intuition by creating several functions on the cat mesh and visualising them in polyscope.

In [3]:
# ── Coordinate functions ────────────────────────────────────────────────────
f_x = V[:, 0]  # x-coordinate at each vertex
f_y = V[:, 1]
f_z = V[:, 2]  # height

# ── Distance from a chosen source vertex ───────────────────────────────────
src = 0  # vertex index of the source
f_dist = np.linalg.norm(V - V[src], axis=1)  # Euclidean distance (proxy for geodesic)

# ── A smooth oscillating function ──────────────────────────────────────────
f_sin = np.sin(8 * np.pi * f_z / (f_z.max() - f_z.min()))  # sinusoid along height

# ── A rough indicator function ─────────────────────────────────────────────
# 1 for the top half of the cat, 0 for the bottom
f_top = (f_z > f_z.mean()).astype(float)

In [4]:
ps_cat = ps.register_surface_mesh("cat", V, F, smooth_shade=True)
ps_cat.add_scalar_quantity("f_x  (x-coordinate)", f_x, enabled=False, cmap="phase")
ps_cat.add_scalar_quantity("f_z  (height)", f_z, enabled=True, cmap="viridis")
ps_cat.add_scalar_quantity(
    "f_dist (Euclidean from v_0)", f_dist, enabled=False, cmap="reds"
)
ps_cat.add_scalar_quantity("f_sin (oscillating)", f_sin, enabled=False, cmap="coolwarm")
ps_cat.add_scalar_quantity("f_top (indicator)", f_top, enabled=False, cmap="blues")

ps.show()  # switch between quantities in the GUI

Observe:

- `f_z` and `f_x` vary **smoothly** — the colour changes gradually across the surface.
- `f_sin` oscillates: it is smooth locally but alternates sign globally.
- `f_top` is **discontinuous**: it jumps abruptly at the equator.
- `f_dist` has concentric rings around vertex 0.

All of these are valid functions. What changes is their *regularity*, which we will quantify using the Dirichlet energy in notebook 03.

## Step 2 — the function space $L^2(M)$

### The continuous picture

A smooth surface $M$ carries a **Riemannian metric** $g$ inherited from $\mathbb{R}^3$. The metric defines a notion of area: every infinitesimal patch $dp$ around a point $p$ has area $dA_p$.

The **$L^2$ space** of square-integrable functions on $M$ is:

$$L^2(M) = \left\{\, f : M \to \mathbb{R} \;\middle|\; \int_M f(x)^2 \, dA < \infty \,\right\}$$

This is a **Hilbert space** — an infinite-dimensional complete inner product space. The inner product is:

$$\langle f, g \rangle_{L^2(M)} = \int_M f(x)\, g(x)\, dA$$

It measures how much $f$ and $g$ *co-vary* over the surface, weighted by the area element $dA$.

The induced **$L^2$ norm** is:

$$\|f\|_{L^2(M)} = \sqrt{\langle f, f \rangle_{L^2(M)}} = \sqrt{\int_M f(x)^2\, dA}$$

Two functions are **orthogonal** if $\langle f, g \rangle_{L^2(M)} = 0$.

### The discrete approximation

On a mesh with $n$ vertices, we approximate the integral by a sum. Each vertex $v_i$ represents a patch of the surface of area $a_i$ (one third of its adjacent triangle areas). Thus:

$$\int_M f(x)\, dA \approx \sum_{i=1}^n f_i\, a_i = \mathbf{f}^\top \mathbf{a}$$

and the inner product becomes:

$$\langle f, g \rangle_M = \sum_{i=1}^n f_i\, g_i\, a_i = \mathbf{f}^\top M\, \mathbf{g}$$

where $M = \mathrm{diag}(a_1, \ldots, a_n)$ is the diagonal **mass matrix**.

The norm and orthogonality conditions become:

$$\|f\|_M = \sqrt{\mathbf{f}^\top M\, \mathbf{f}}, \qquad f \perp_M g \iff \mathbf{f}^\top M\, \mathbf{g} = 0$$

**Why not just use the standard dot product?**  
The plain dot product $\mathbf{f} \cdot \mathbf{g} = \sum_i f_i g_i$ treats all vertices equally. But vertices in densely sampled regions contribute more samples — without the area weighting, dense regions dominate the integral. The mass matrix $M$ exactly corrects for this.

## Step 3 — compute inner products and norms

In [11]:
# The mass matrix M is the core geomfum object encoding vertex areas.
# It is a sparse diagonal matrix: M = diag(a_1, ..., a_n).
# Accessing it triggers the Laplacian computation the first time.
M = mesh.laplacian.mass_matrix  # sparse (n, n)

print(
    f"Total surface area = sum of vertex areas = {np.asarray(mesh.vertex_areas).sum():.4f}"
)
print(f"Mass matrix type: {type(M).__name__}")


# ── Inner product  <f, g>_M = f^T M g ──────────────────────────────────────
# la.matvecmul handles the sparse M @ dense v product natively.
def ip(f, g):
    """Area-weighted L2 inner product using geomfum's matvecmul."""
    Mg = la.matvecmul(M, g)  # M @ g  →  (n,)
    return float(f @ Mg)


# ── Norm  ||f||_M = sqrt(f^T M f) ──────────────────────────────────────────
def norm_M(f):
    """Area-weighted L2 norm using geomfum's matvecmul."""
    return float(np.sqrt(f @ la.matvecmul(M, f)))

Total surface area = sum of vertex areas = 0.3502
Mass matrix type: csc_matrix


In [12]:
# Norms of our functions
for name, f in [
    ("f_x", f_x),
    ("f_z", f_z),
    ("f_sin", f_sin),
    ("f_top", f_top),
    ("f_dist", f_dist),
]:
    print(f"  ||{name}||_M = {norm_M(f):.5f}")

  ||f_x||_M = 0.02551
  ||f_z||_M = 0.10688
  ||f_sin||_M = 0.41802
  ||f_top||_M = 0.38873
  ||f_dist||_M = 0.17120


In [13]:
# Are the coordinate functions orthogonal?
print("Inner products between coordinate functions:")
for (na, fa), (nb, fb) in [
    (("x", f_x), ("y", f_y)),
    (("x", f_x), ("z", f_z)),
    (("y", f_y), ("z", f_z)),
]:
    print(f"  <f_{na}, f_{nb}>_M = {ip(fa, fb):.6f}")
print()
print("Note: coordinate functions are NOT necessarily orthogonal —")
print("orthogonality depends on the shape's geometry.")

Inner products between coordinate functions:
  <f_x, f_y>_M = 0.000021
  <f_x, f_z>_M = -0.000021
  <f_y, f_z>_M = 0.003388

Note: coordinate functions are NOT necessarily orthogonal —
orthogonality depends on the shape's geometry.


In [15]:
# ── M-weighted normalisation ────────────────────────────────────────────────
# Divide by ||f||_M  so the result has unit mass-weighted norm.
f_z_hat = f_z / norm_M(f_z)  # element-wise division

print(f"||f_z_hat||_M = {norm_M(f_z_hat):.10f}  (should be 1.0)")

# ── Contrast: la.normalize  (unweighted L2 norm, NOT mass-weighted) ─────────
# geomfum.linalg.normalize divides by the standard Euclidean norm  ||f||_2,
# which ignores vertex areas.  Useful for normalising coefficient vectors,
# NOT for functions in L^2(M).
f_z_l2 = la.normalize(f_z, axis=0)  # divides by  sqrt(sum f_i^2)
print(
    f"||f_z_l2||_2  = {float(gs.linalg.norm(f_z_l2)):.10f}  (unit L2 norm, unweighted)"
)
print(f"||f_z_l2||_M  = {norm_M(f_z_l2):.6f}  (NOT 1 — different norm)")

||f_z_hat||_M = 1.0000000000  (should be 1.0)
||f_z_l2||_2  = 1.0000000000  (unit L2 norm, unweighted)
||f_z_l2||_M  = 0.006280  (NOT 1 — different norm)


## Step 4 — distance between functions

The $L^2$ norm immediately gives a notion of **distance** between two functions:

$$d_M(f, g) = \|f - g\|_M = \sqrt{\langle f - g,\, f - g \rangle_M}$$

This quantifies how different two scalar fields are *over the whole surface*, properly weighted by area. It will be the basis for measuring how well we reconstruct a function from a truncated basis (notebook 02) and how accurately a functional map transfers signals (notebook 08).

In [16]:
def dist_M(f, g):
    """L2 distance between two functions."""
    return norm_M(f - g)


# f_sin and f_top are both 'alternating' — are they close?
print(f"d_M(f_z,   f_x)   = {dist_M(f_z, f_x):.5f}")
print(f"d_M(f_sin, f_top) = {dist_M(f_sin, f_top):.5f}")
print(f"d_M(f_z,   f_z)   = {dist_M(f_z, f_z):.5f}  (self-distance = 0)")

d_M(f_z,   f_x)   = 0.11007
d_M(f_sin, f_top) = 0.64290
d_M(f_z,   f_z)   = 0.00000  (self-distance = 0)


In [17]:
# Visualise the difference f_sin - f_top
diff = f_sin - f_top

ps.remove_all_structures()
ps_cat = ps.register_surface_mesh("cat", V, F, smooth_shade=True)
ps_cat.add_scalar_quantity("f_sin", f_sin, enabled=False, cmap="RdBu")
ps_cat.add_scalar_quantity("f_top", f_top, enabled=False, cmap="blues")
ps_cat.add_scalar_quantity("f_sin - f_top", diff, enabled=True, cmap="coolwarm")

ps.show()

## Where to go next

- [03 — What is a Basis?](./03_what_is_a_basis.ipynb): represent any function as a sum of basis elements
- [04 — Tangent Vectors and Differential Operators](./04_tangent_vectors_and_operators.ipynb): measure smoothness via the gradient
- [How to compute the Laplacian?](../how_to/01_mesh_laplacian.ipynb)